# Qwen Taboo J-Lens: environment smoke test

**Objective:** verify the persistent kernel, GPU runtime, package versions, project paths, and small Hugging Face metadata before downloading Qwen3.6-27B weights.

**Success criteria:** CUDA is available on one suitable GPU; the report is saved; model, adapter, and `_n1000` J-Lens metadata pass `scripts/verify_artifacts.py`. This notebook must not load the 27B model.


In [ ]:
# Reproducible project paths and imports
from __future__ import annotations

import json
import random
import subprocess
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

SEED = 7
random.seed(SEED)
PROJECT_ROOT


## Plan

1. Collect environment and GPU metadata.
2. Confirm the project smoke-test configuration.
3. Run metadata-only Hugging Face artifact checks.
4. Save reports and stop for review before large downloads.


In [ ]:
# Collect and save the lightweight environment report
from src.environment_report import save_environment

environment = save_environment(PROJECT_ROOT / 'results/environment_report.json')
{
    'python': environment['python'].split()[0],
    'packages': environment['packages'],
    'torch_runtime': environment.get('torch_runtime'),
    'torch_runtime_error': environment.get('torch_runtime_error'),
}


In [ ]:
# Fail early if the primary GPU requirement is not met
runtime = environment.get('torch_runtime', {})
assert runtime.get('cuda_available'), 'CUDA is unavailable; stop before downloading weights.'
gpu_gib = [round(d['total_memory_bytes'] / 2**30, 1) for d in runtime.get('devices', [])]
print({'devices': runtime.get('devices'), 'memory_gib': gpu_gib})
if not any(memory >= 75 for memory in gpu_gib):
    print('WARNING: no approximately 80 GB GPU detected. Do not silently quantize or CPU-offload.')


In [ ]:
# Inspect the declared condition without loading any weights
config_path = PROJECT_ROOT / 'configs/smoke_test.json'
config = json.loads(config_path.read_text())
config


In [ ]:
# Metadata-only preflight: downloads small JSON configs, not model weights
completed = subprocess.run(
    [sys.executable, str(PROJECT_ROOT / 'scripts/verify_artifacts.py')],
    cwd=PROJECT_ROOT,
    text=True,
    capture_output=True,
)
print(completed.stdout)
if completed.returncode != 0:
    print(completed.stderr)
    raise RuntimeError('Artifact preflight failed; inspect results/artifact_preflight.json')
artifact_report = json.loads((PROJECT_ROOT / 'results/artifact_preflight.json').read_text())
artifact_report


## Review gate

Before downloading large weights, inspect both JSON reports and record resolved SHAs in `research_log.md`. Confirm that the adapter names the intended base model and that the exact `_n1000` J-Lens file exists. Then show the proposed model-loading command and estimated download/storage requirement for approval.


## Next steps

- If every check passes: create a separate behavioral smoke notebook or script for one published `gold` example.
- If metadata mismatches: stop and update the artifact decision; do not force-load.
- If Jupyter MCP fails: time-box debugging to 30–45 minutes, then use the documented `tmux`/IPython fallback.
